# 복지마을 방송국 (Welfare Village Broadcaster) — v2

**LangGraph + 공공데이터 2종 + Tavily 멀티에이전트**

> *"복지는 신청하는 사람이 아니라, 필요한 사람에게 먼저 찾아가야 합니다."*

## 기획서 반영 사항 (v1 → v2)
- **1차 데이터**: 행정안전부 `대한민국 공공서비스(혜택)` API + 한국사회보장정보원 `지자체 복지서비스` API
- **2차 검증**: Tavily 웹검색(복지로·정부24 우선) — API 누락된 지자체 한시·신규 사업 보완
- **자격 매칭**: 공공API `supportConditions` 응답을 규칙 기반으로 매칭 → **LLM 환각 없는 판단**
- **LLM 역할 제한**: 쉬운말 변환·상담·방송 스크립트 생성에만 사용

## 데이터 흐름

```
사용자 입력 → [Supervisor]
              ├→ [welfare_search] ── ① 공공서비스 API (api.odcloud.kr/gov24/v3)
              │                   ── ② 지자체복지 API (apis.data.go.kr/B554287)
              │                   ── ③ Tavily 보완 검색
              ├→ [eligibility_check] ── supportConditions → 규칙 매칭
              ├→ [easy_translate]   ── LLM(쉬운말, 사투리)
              ├→ [broadcast_script] ── LLM(TTS용 방송 스크립트)
              ├→ [qna_agent]        ── LLM(전화 상담 페르소나)
              └→ END(done)
```


## 1. 환경 설정 & API 키 확인

`.env.example`을 `.env`로 복사 후 5개 키를 채워주세요. 자세한 발급법은 README 참고.


In [ ]:
# !pip install -r requirements.txt


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

REQUIRED = ["PUBLIC_SERVICE_API_KEY", "LOCAL_WELFARE_API_KEY", "OPENAI_API_KEY"]
missing = [k for k in REQUIRED if not os.getenv(k) or os.getenv(k).startswith("발급") or os.getenv(k).startswith("sk-proj-...")]
if missing:
    print(f"⚠️ 다음 키가 비어있거나 placeholder입니다: {missing}")
    print("   .env 파일을 확인하세요.")
else:
    print("✅ 필수 키 모두 확인됨")

HAS_TAVILY = bool(os.getenv("TAVILY_API_KEY") and not os.getenv("TAVILY_API_KEY","").startswith("tvly-..."))
USE_LANGFUSE = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
print(f"Tavily 사용: {HAS_TAVILY}  |  LangFuse 사용: {USE_LANGFUSE}")


## 2. 공공데이터 API 클라이언트

### 2-1. 행정안전부 공공서비스(혜택) API — `api.odcloud.kr/api/gov24/v3`
- `/serviceList` — 서비스명/소관기관명/사용자구분/서비스분야로 검색
- `/serviceDetail` — 서비스ID로 상세 조회
- `/supportConditions` — 서비스ID로 지원조건(연령·소득·가구형태·장애·보훈 등) 조회

### 2-2. 한국사회보장정보원 지자체 복지서비스 API — `apis.data.go.kr/B554287/LocalGovernmentWelfareInformations`
- `/LcgvWelfarelist` — 지자체 복지서비스 목록 (XML)
- `/LcgvWelfaredetailed` — 상세 조회 (XML)


In [ ]:
import requests
import xml.etree.ElementTree as ET
from typing import List, Dict, Any, Optional
from urllib.parse import unquote

PUBLIC_API_BASE = "https://api.odcloud.kr/api/gov24/v3"
LOCAL_API_BASE = "https://apis.data.go.kr/B554287/LocalGovernmentWelfareInformations"

def _public_key():
    """data.go.kr 인증키 - Encoding 상태로 .env에 저장돼 있으면 requests가 다시 인코딩하지 않도록 디코딩"""
    raw = os.getenv("PUBLIC_SERVICE_API_KEY", "")
    return unquote(raw) if "%" in raw else raw

def _local_key():
    raw = os.getenv("LOCAL_WELFARE_API_KEY", "")
    return unquote(raw) if "%" in raw else raw

# ---------- 행정안전부 공공서비스 ----------
def call_public_service_list(keyword: str = "", target_user: str = "", category: str = "", per_page: int = 20) -> List[Dict]:
    """공공서비스 목록 조회 (LIKE 검색)"""
    params = {
        "serviceKey": _public_key(),
        "page": 1, "perPage": per_page, "returnType": "JSON",
    }
    if keyword: params["cond[서비스명::LIKE]"] = keyword
    if target_user: params["cond[사용자구분::LIKE]"] = target_user
    if category: params["cond[서비스분야::LIKE]"] = category
    r = requests.get(f"{PUBLIC_API_BASE}/serviceList", params=params, timeout=15)
    r.raise_for_status()
    return r.json().get("data", [])

def call_public_support_conditions(service_id: str) -> Dict:
    """특정 서비스의 지원조건 조회 (자격 매칭에 사용)"""
    params = {
        "serviceKey": _public_key(),
        "page": 1, "perPage": 1, "returnType": "JSON",
        "cond[서비스ID::EQ]": service_id,
    }
    r = requests.get(f"{PUBLIC_API_BASE}/supportConditions", params=params, timeout=15)
    r.raise_for_status()
    data = r.json().get("data", [])
    return data[0] if data else {}

# ---------- 한국사회보장정보원 지자체 복지 ----------
def call_local_welfare_list(region: str = "", keyword: str = "", num_rows: int = 20) -> List[Dict]:
    """지자체 복지서비스 목록 조회 (XML 응답 → dict 변환)"""
    params = {
        "serviceKey": _local_key(),
        "pageNo": 1, "numOfRows": num_rows,
    }
    if region: params["ctpvNm"] = region
    if keyword: params["searchKeyword"] = keyword
    r = requests.get(f"{LOCAL_API_BASE}/LcgvWelfarelist", params=params, timeout=15)
    r.raise_for_status()
    items = []
    try:
        root = ET.fromstring(r.text)
        for item in root.findall(".//servList") + root.findall(".//item"):
            items.append({child.tag: (child.text or "").strip() for child in item})
    except ET.ParseError as e:
        print(f"⚠️ 지자체 복지 API XML 파싱 실패: {e}")
        print(f"   응답 일부: {r.text[:300]}")
    return items

# ---------- 연결 테스트 ----------
try:
    sample = call_public_service_list(keyword="기초연금", per_page=2)
    print(f"✅ 행정안전부 API OK - {len(sample)}건 샘플:")
    for s in sample[:2]:
        print(f"   - {s.get('서비스명','?')} ({s.get('소관기관명','?')})")
except Exception as e:
    print(f"❌ 행정안전부 API 실패: {e}")

try:
    sample = call_local_welfare_list(region="충청남도", num_rows=2)
    print(f"✅ 지자체복지 API OK - {len(sample)}건 샘플")
    for s in sample[:2]:
        # 필드명은 코드표에 따라 다를 수 있음 - 주요 키만 출력
        title = s.get("servNm") or s.get("서비스명") or list(s.values())[0] if s else "?"
        print(f"   - {title}")
except Exception as e:
    print(f"⚠️ 지자체복지 API 연결 실패 (인증키/네트워크 확인): {e}")


## 3. LLM & Tavily 초기화

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.3)
router_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

tavily_tool = None
if HAS_TAVILY:
    from langchain_tavily import TavilySearch
    tavily_tool = TavilySearch(
        max_results=5, topic="general",
        include_domains=["bokjiro.go.kr", "gov.kr", "mohw.go.kr", "korea.kr"],
        search_depth="advanced",
    )
print("LLM 초기화 완료, Tavily:", "사용" if tavily_tool else "비활성")


## 4. 자격 매칭 (`supportConditions` 기반 규칙 엔진)

공공서비스 API의 `supportConditions` 응답은 각 서비스의 지원조건을 코드(JA****)로 알려줍니다:
- `JA0110/0111`: 대상연령 시작/종료
- `JA0201~JA0205`: 중위소득 구간 (0~50%, 51~75%, 76~100%, 101~200%, 200% 초과)
- `JA0328`: 장애인, `JA0329`: 국가보훈, `JA0330`: 질병/질환자
- `JA0313~JA0316`: 농업인/어업인/축산업인/임업인
- `JA0403`: 한부모/조손, `JA0404`: 1인가구, `JA0411`: 다자녀, `JA0412`: 무주택

이 값들을 사용자 프로필과 매칭해 **LLM 없이** 자격 여부를 판단합니다.


In [ ]:
# 2026 기준 중위소득 1인가구 약 2,392,013원 — 구간별 상한 추정
INCOME_BANDS = [
    ("JA0201", 1_196_000),   # 50%
    ("JA0202", 1_794_000),   # 75%
    ("JA0203", 2_392_000),   # 100%
    ("JA0204", 4_784_000),   # 200%
    ("JA0205", 10**12),      # 200% 초과
]

class UserProfile(BaseModel):
    age: int = Field(description="만 나이")
    region: str = Field(default="", description="거주 시도 (예: '충청남도')")
    monthly_income: Optional[int] = Field(default=None, description="월 소득(원)")
    has_disability: bool = Field(default=False)
    is_rural: bool = Field(default=False, description="농어촌 거주")
    is_single_household: bool = Field(default=False, description="1인가구")
    is_single_parent: bool = Field(default=False, description="한부모/조손가정")

def _truthy(v) -> bool:
    """API 응답이 'Y'/'1'/'true'/특정 문자열로 올 수 있어 관대하게 판정"""
    if v is None: return False
    s = str(v).strip().lower()
    return s in {"y", "1", "true", "t", "해당"}

def is_eligible(profile: UserProfile, conditions: Dict) -> tuple[bool, str]:
    """supportConditions와 사용자 프로필을 매칭. (자격여부, 사유) 반환"""
    if not conditions:
        return True, "지원조건 정보 없음 (조회 후 안내 필요)"

    # 연령
    age_s = conditions.get("JA0110")
    age_e = conditions.get("JA0111")
    if age_s:
        try:
            if profile.age < int(age_s): return False, f"연령 {age_s}세 이상 대상"
        except (TypeError, ValueError): pass
    if age_e:
        try:
            if profile.age > int(age_e): return False, f"연령 {age_e}세 이하 대상"
        except (TypeError, ValueError): pass

    # 소득 구간 - 어떤 구간이라도 'Y'로 표시된 게 있고, 사용자 소득이 그 구간 이하면 OK
    income_codes_present = [c for c, _ in INCOME_BANDS if _truthy(conditions.get(c))]
    if income_codes_present and profile.monthly_income is not None:
        ok = False
        for code, limit in INCOME_BANDS:
            if _truthy(conditions.get(code)) and profile.monthly_income <= limit:
                ok = True; break
        if not ok:
            return False, "지원 소득 구간 초과"

    # 특수 조건 - API가 명시한 경우에만 거름
    if _truthy(conditions.get("JA0328")) and not profile.has_disability:
        return False, "장애인 대상"
    if _truthy(conditions.get("JA0403")) and not profile.is_single_parent:
        return False, "한부모/조손가정 대상"
    if _truthy(conditions.get("JA0404")) and not profile.is_single_household:
        return False, "1인가구 대상"

    return True, "자격 매칭"

print("자격 매칭 엔진 준비 완료")


## 5. State & 노드 정의

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

class WelfareState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_profile: Dict[str, Any]
    raw_services: List[Dict[str, Any]]       # 1차 API 원본
    eligible_benefits: List[Dict[str, Any]]  # 자격 매칭된 결과
    easy_text: str
    broadcast_text: str
    next_action: str

# ---------- Supervisor ----------
class RouteDecision(BaseModel):
    next: Literal["search", "eligibility", "easy", "broadcast", "qna", "done"]
    reason: str

SUPERVISOR_PROMPT = """당신은 '복지마을 방송국' 시스템의 매니저입니다.
사용자 메시지와 현재 상태를 보고 다음 중 하나를 선택하세요:
- search: 복지 정보를 공공API+Tavily에서 가져와야 할 때
- eligibility: 사용자 자격(나이/소득/지역)으로 받을 수 있는 복지를 추려야 할 때
- easy: 행정 공지문을 어르신 친화 말투로 변환
- broadcast: 마을 방송용 TTS 스크립트 생성
- qna: 전화 상담처럼 대화 응답
- done: 충분히 해결됨, 종료
이미 결과가 있으면 done을 고르세요."""

def supervisor_node(state: WelfareState) -> dict:
    structured = router_llm.with_structured_output(RouteDecision)
    summary = (
        f"검색결과 {len(state.get('raw_services',[]))}건, "
        f"자격매칭 {len(state.get('eligible_benefits',[]))}건, "
        f"쉬운말 {'있음' if state.get('easy_text') else '없음'}, "
        f"방송 {'있음' if state.get('broadcast_text') else '없음'}"
    )
    decision = structured.invoke([
        SystemMessage(content=SUPERVISOR_PROMPT),
        SystemMessage(content=f"현재 상태: {summary}"),
        *state["messages"][-4:],
    ])
    return {"next_action": decision.next,
            "messages": [AIMessage(content=f"[Supervisor] {decision.next} — {decision.reason}")]}

# ---------- 복지 검색 (1차 공공API + 2차 Tavily) ----------
def welfare_search_node(state: WelfareState) -> dict:
    last_user = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    query = last_user.content if last_user else ""
    profile = state.get("user_profile") or {}

    services = []

    # 키워드 추출 (간단 휴리스틱: LLM 한 번 호출해도 됨)
    keywords = []
    for kw in ["기초연금", "에너지", "노인", "장애", "주거", "일자리", "돌봄", "의료", "교육", "양육"]:
        if kw in query: keywords.append(kw)
    main_kw = keywords[0] if keywords else ""

    # ① 행정안전부 공공서비스
    try:
        public_results = call_public_service_list(keyword=main_kw, per_page=20)
        for s in public_results:
            services.append({
                "source": "행정안전부", "id": s.get("서비스ID"),
                "name": s.get("서비스명"), "agency": s.get("소관기관명"),
                "target": s.get("지원대상"), "content": s.get("지원내용"),
                "apply": s.get("신청방법"), "url": s.get("상세조회URL"),
            })
    except Exception as e:
        print(f"공공서비스 API 오류: {e}")

    # ② 지자체 복지
    try:
        region = profile.get("region", "").split()[0] if profile.get("region") else ""  # 시도만
        local_results = call_local_welfare_list(region=region, keyword=main_kw, num_rows=20)
        for s in local_results:
            services.append({
                "source": "지자체", "id": s.get("servId") or s.get("서비스ID"),
                "name": s.get("servNm") or s.get("서비스명"),
                "agency": s.get("ctpvNm","") + " " + s.get("sggNm",""),
                "target": s.get("trgterIndvdlArray") or s.get("지원대상",""),
                "content": s.get("servDgst","") or s.get("지원내용",""),
                "apply": s.get("aplyMtdCn","") or s.get("신청방법",""),
                "url": s.get("servDtlLink","") or "",
            })
    except Exception as e:
        print(f"지자체복지 API 오류: {e}")

    # ③ Tavily 보완
    if tavily_tool and main_kw:
        try:
            tv_q = f"{main_kw} {profile.get('region','')} 복지 지원 신청"
            raw = tavily_tool.invoke({"query": tv_q})
            tv_results = raw.get("results", []) if isinstance(raw, dict) else raw
            for r in tv_results[:3]:
                services.append({
                    "source": "Tavily(웹)", "id": None,
                    "name": r.get("title",""), "agency": "웹검색",
                    "target": "", "content": r.get("content",""),
                    "apply": "", "url": r.get("url",""),
                })
        except Exception as e:
            print(f"Tavily 오류: {e}")

    # 중복 제거 (서비스명 기준)
    seen = set(); unique = []
    for s in services:
        nm = s.get("name","")
        if nm and nm not in seen:
            seen.add(nm); unique.append(s)

    summary_lines = [f"- [{s['source']}] {s['name']} ({s.get('agency','')})" for s in unique[:10]]
    return {
        "raw_services": unique,
        "messages": [AIMessage(content=f"🔎 1차+2차 검색 결과 총 {len(unique)}건:\n" + "\n".join(summary_lines))],
    }

# ---------- 자격 매칭 ----------
def eligibility_node(state: WelfareState) -> dict:
    profile_dict = state.get("user_profile") or {}
    if not profile_dict.get("age"):
        return {"messages": [AIMessage(content="자격 확인을 위해 어르신의 나이/지역/소득 정보가 필요합니다.")]}
    profile = UserProfile(**{k:v for k,v in profile_dict.items() if k in UserProfile.model_fields})

    services = state.get("raw_services", [])
    if not services:
        # 자격 확인 전 검색이 안 됐다면 직접 한 번 호출
        try:
            services = []
            for kw in ["노인", "기초연금", "에너지"]:
                services.extend(call_public_service_list(target_user=kw, per_page=10))
            services = [{"source":"행정안전부","id":s.get("서비스ID"),"name":s.get("서비스명"),
                         "agency":s.get("소관기관명"),"target":s.get("지원대상"),
                         "content":s.get("지원내용"),"apply":s.get("신청방법"),"url":s.get("상세조회URL")}
                        for s in services]
        except Exception as e:
            return {"messages":[AIMessage(content=f"검색 실패: {e}")]}

    matched = []
    for s in services:
        # 공공API 서비스만 supportConditions로 정밀 매칭, 나머지는 LLM 없이 패스(보여만 줌)
        if s.get("source") == "행정안전부" and s.get("id"):
            try:
                cond = call_public_support_conditions(s["id"])
                ok, reason = is_eligible(profile, cond)
                if ok:
                    matched.append({**s, "match_reason": reason})
            except Exception:
                continue
        else:
            matched.append({**s, "match_reason": "지자체/웹 검색 결과(추가 확인 필요)"})

    if not matched:
        msg = "조건에 맞는 복지를 찾지 못했습니다. 다른 조건으로 다시 시도해보세요."
    else:
        lines = [f"- ✅ **{m['name']}** ({m.get('agency','')})\n  → {m.get('content','')[:120]}..." for m in matched[:8]]
        msg = f"✅ 신청 가능 복지 {len(matched)}건:\n" + "\n".join(lines)
    return {"eligible_benefits": matched, "messages": [AIMessage(content=msg)]}

# ---------- 쉬운말 변환 ----------
EASY_PROMPT = """당신은 행정 공지를 시골 어르신도 알아듣게 풀어주는 친근한 동네 통장님입니다.
규칙:
1) 한자어/외래어 줄이기 2) 짧은 문장(15자 내외) 3) 따뜻한 어투, 어르신 호칭
4) 신청 방법/장소/필요 서류 명확히 5) dialect=='chungcheong' 충청도, 'jeolla' 전라도, 그 외 표준어"""

def easy_translate_node(state: WelfareState) -> dict:
    last_user = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last_user:
        return {"messages":[AIMessage(content="변환할 원문이 없습니다.")]}
    dialect = (state.get("user_profile") or {}).get("dialect","standard")
    out = llm.invoke([
        SystemMessage(content=EASY_PROMPT + f"\nsdialect: {dialect}"),
        HumanMessage(content=f"다음 공지를 변환:\n\n{last_user.content}"),
    ])
    return {"easy_text": out.content,
            "messages":[AIMessage(content=f"📣 쉬운 말 변환:\n{out.content}")]}

# ---------- 방송 스크립트 ----------
BROADCAST_PROMPT = """마을 스피커 방송용 멘트를 짭니다. 구조:
1) 인삿말 ("어르신 안녕하세요...") 2) 핵심 안내 3) 대상자 4) 신청 방법(장소/일자/서류) 5) 마무리.
분량: 30초 (한국어 90-130자). 따뜻하고 천천히."""

def broadcast_node(state: WelfareState) -> dict:
    base = state.get("easy_text", "")
    benefits = state.get("eligible_benefits", [])
    last_user = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    src = base or (last_user.content if last_user else "")
    if benefits:
        src += "\n\n관련 복지: " + ", ".join([b["name"] for b in benefits[:5]])
    out = llm.invoke([SystemMessage(content=BROADCAST_PROMPT), HumanMessage(content=f"원문:\n{src}")])
    return {"broadcast_text": out.content,
            "messages":[AIMessage(content=f"📻 마을 방송 스크립트:\n{out.content}")]}

# ---------- Q&A 페르소나 ----------
QNA_PROMPT = """전화 상담원입니다. 천천히, 짧은 문장으로. 어려운 용어는 그때그때 풀어 설명.
모르면 '면사무소에 전화 한 통 넣어드릴게요'. 마지막은 항상 '더 궁금하신 점 있으세요?'.
state의 eligible_benefits를 적극 활용."""

def qna_node(state: WelfareState) -> dict:
    ctx = ""
    if state.get("eligible_benefits"):
        ctx = "\n자격 가능: " + ", ".join([b["name"] for b in state["eligible_benefits"][:5]])
    out = llm.invoke([SystemMessage(content=QNA_PROMPT + ctx), *state["messages"][-6:]])
    return {"messages":[AIMessage(content=out.content)]}

print("6개 노드 정의 완료")


## 6. 그래프 빌드

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

def route(state: WelfareState):
    nxt = state.get("next_action","qna")
    if nxt == "done": return END
    return {"search":"welfare_search","eligibility":"eligibility_check",
            "easy":"easy_translate","broadcast":"broadcast_script","qna":"qna_agent"}.get(nxt,"qna_agent")

builder = StateGraph(WelfareState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("welfare_search", welfare_search_node)
builder.add_node("eligibility_check", eligibility_node)
builder.add_node("easy_translate", easy_translate_node)
builder.add_node("broadcast_script", broadcast_node)
builder.add_node("qna_agent", qna_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route, {
    "welfare_search":"welfare_search","eligibility_check":"eligibility_check",
    "easy_translate":"easy_translate","broadcast_script":"broadcast_script",
    "qna_agent":"qna_agent", END:END,
})
for n in ["welfare_search","eligibility_check","easy_translate","broadcast_script","qna_agent"]:
    builder.add_edge(n, "supervisor")

graph = builder.compile(checkpointer=InMemorySaver())
print("그래프 컴파일 완료")


## 7. 그래프 시각화

In [ ]:
graph


## 8. LangFuse 콜백 (옵션)

In [ ]:
langfuse_handler = None
if USE_LANGFUSE:
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print("LangFuse 활성")
else:
    print("LangFuse 비활성 (정상 동작에 영향 없음)")

def cfg(thread_id: str):
    c = {"configurable":{"thread_id":thread_id}}
    if langfuse_handler: c["callbacks"] = [langfuse_handler]
    return c


## 9. 시연 1 — 자격 확인 (충남 부여 78세 어르신)

In [ ]:
profile_grandma = {
    "age": 78, "region": "충청남도 부여군",
    "monthly_income": 800_000,
    "has_disability": False, "is_rural": True,
    "is_single_household": True, "is_single_parent": False,
    "dialect": "chungcheong",
}

state1 = {
    "messages": [HumanMessage(content="이장님~ 나도 받을 수 있는 노인 복지가 뭐가 있나 알아봐주세요.")],
    "user_profile": profile_grandma,
    "raw_services": [], "eligible_benefits": [],
    "easy_text": "", "broadcast_text": "", "next_action": "",
}

result = graph.invoke(state1, config=cfg("demo-grandma-1"))
print("=== 최종 메시지 ===")
for m in result["messages"][-4:]:
    role = m.__class__.__name__.replace("Message","")
    print(f"[{role}] {m.content[:500]}\n")


## 10. 시연 2 — 행정 공지문 → 어르신 친화 마을방송 (충청도 사투리)

In [ ]:
official_notice = """기초연금 신청 기간이 도래하였으니, 만 65세 이상 거주민께서는 신분증 및 통장 사본을 지참하시어 6월 15일까지 행정복지센터를 방문 바랍니다. 미신청 시 해당 분기 지급이 누락될 수 있습니다."""

state2 = {
    "messages": [
        HumanMessage(content=official_notice),
        HumanMessage(content="이걸 어르신 알아듣게 풀어주시고, 마을 방송 멘트로도 만들어주세요."),
    ],
    "user_profile": {"dialect": "chungcheong"},
    "raw_services": [], "eligible_benefits": [],
    "easy_text": "", "broadcast_text": "", "next_action": "",
}
result2 = graph.invoke(state2, config=cfg("demo-broadcast-1"))
print("=== 쉬운 말 ===\n", result2.get("easy_text") or "(없음)")
print("\n=== 마을 방송 ===\n", result2.get("broadcast_text") or "(없음)")


## 11. 시연 3 — 멀티턴 전화 상담 (같은 thread_id로 컨텍스트 이어짐)

In [ ]:
thread = "demo-phone-1"

r1 = graph.invoke(
    {"messages":[HumanMessage(content="여보세요? 나 올해 78인디, 에너지바우처라는거 받을 수 있나?")],
     "user_profile":{"age":78,"monthly_income":800_000,"is_rural":True,"region":"충남 부여",
                     "is_single_household":True, "has_disability":False, "is_single_parent":False},
     "raw_services":[],"eligible_benefits":[],"easy_text":"","broadcast_text":"","next_action":""},
    config=cfg(thread),
)
print("[1턴]", r1["messages"][-1].content[:500], "\n")

r2 = graph.invoke(
    {"messages":[HumanMessage(content="신청은 어디로 가야 하나요? 뭘 가져가야 해요?")]},
    config=cfg(thread),
)
print("[2턴]", r2["messages"][-1].content[:500])


## 12. 다음 단계 (해커톤 확장)

1. **TTS 연동** — `easy_text`/`broadcast_text`를 CLOVA Dubbing·ElevenLabs로 mp3 출력
2. **STT 입력** — 전화 음성 → Whisper → `graph.invoke`
3. **HITL** — SMS/카톡 발송 노드 앞에 `HumanInTheLoopMiddleware`
4. **장기 메모리** — `SqliteSaver` + `InMemoryStore`로 어르신 프로필 영구 저장
5. **위험 감지** — N일간 미응답 어르신 → 복지사 알림
6. **MCP 서버화** — 공공API 도구를 FastMCP로 분리 → Cursor/Claude Desktop 재사용

> "복지의 마지막 한 걸음은 '신청'이 아니라 **'도달'** 이다."
